In [1]:
import requests

api_url = "https://115.159.159.82/api/devices"

r = requests.get(
    api_url,
    params={
        "page":1,"page_size":50
    },
    timeout=10
)

print("status:", r.status_code)
print("url:", r.url)
print("content-type:", r.headers.get("Content-Type"))
data = r.json()

print(type(data))
print(data.keys())

status: 200
url: https://115.159.159.82/api/devices?page=1&page_size=50
content-type: application/json
<class 'dict'>
dict_keys(['page', 'page_size', 'total', 'items'])


In [2]:
print(data)

{'page': 1, 'page_size': 50, 'total': 5000, 'items': [{'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}, {'device_id': 'D00000002', 'company_id': 'C000001', 'device_name': '循环泵-000001-002', 'device_type': 'pump', 'model': 'SP-PUM-500', 'zone': '智造一区', 'install_date': '2016-03-06', 'rated_power_kw': 73.8, 'sampling_interval_s': 60, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-08T05:44:00'}, {'device_id': 'D00000003', 'company_id': 'C000001', 'device_name': '空压机-000001-003', 'device_type': 'compressor', 'model': 'SP-COM-200', 'zone': '智造一区', 'install_date': '2016-12-18', 'rated_power_kw': 248.6, 'sampling_interval_s': 30, 'protocol': 'modbus', 'status': 'running', 'updated_at': '2026-08-29T04:01:00'}, {'device_id': 'D

In [3]:
print(data["items"][0])

{'device_id': 'D00000001', 'company_id': 'C000001', 'device_name': '循环泵-000001-001', 'device_type': 'pump', 'model': 'SP-PUM-800', 'zone': '智造一区', 'install_date': '2021-09-19', 'rated_power_kw': 71.2, 'sampling_interval_s': 30, 'protocol': 'mqtt', 'status': 'running', 'updated_at': '2026-09-01T03:58:00'}


In [4]:
total = data["total"]
page_size = data["page_size"]

total_pages = (total + page_size - 1) // page_size

print("total:", total)
print("page_size:", page_size)
print("total_pages:", total_pages)

total: 5000
page_size: 50
total_pages: 100


In [9]:
devices = []

for page in range(1, total_pages + 1):
    r = requests.get(
        api_url,
        params={
            "page": page,
            "page_size": page_size,
        },
        timeout=10
    )

    page_data = r.json()

    # 将本页设备加入 devices
    devices.extend(page_data["items"])

print("device count:", len(devices))

device count: 5000


In [10]:
print("first:", devices[0]["device_id"])
print("last :", devices[-1]["device_id"])

first: D00000001
last : D00005000


In [12]:
from bs4 import BeautifulSoup

base_url = "https://115.159.159.82"
current_url = base_url + "/companies"

r = requests.get(current_url, timeout=10)
soup = BeautifulSoup(r.text, "html.parser")

In [13]:
cards = soup.select("article.company-card")

print("本页企业数：", len(cards))

本页企业数： 20


In [16]:
card = cards[0]

In [17]:
details = {}

for item in card.select("dl > div"):
    key = item.select_one("dt").get_text(strip=True)
    value = item.select_one("dd").get_text(strip=True)
    details[key] = value

print(details)

{'所属行业': '汽车零部件', '所在区域': '智造一区', '员工规模': '88 人', '设备数量': '6 台'}


In [19]:
company_id = card["data-company-id"]                        # 来自 HTML 属性 data-*
name = card.select_one("h2 a").get_text(strip=True)         # 来自文本
status = card.select_one(".badge").get_text(strip=True)     # 来自文本

industry = details["所属行业"]
zone = details["所在区域"]
device_count = int(details["设备数量"].replace("台", "").strip())   # "6 台" → 6

print("company_id  :", company_id)
print("name        :", name)
print("industry    :", industry)
print("zone        :", zone)
print("device_count:", device_count)
print("status      :", status)

company_id  : C000001
name        : 临港精工001有限公司
industry    : 汽车零部件
zone        : 智造一区
device_count: 6
status      : 运营中


In [22]:
def parse_company_card(card):
    """从一条 .company-card 节点中提取企业字段。"""
    details = {}
    for item in card.select("dl > div"):
        key = item.select_one("dt").get_text(strip=True)
        value = item.select_one("dd").get_text(strip=True)
        details[key] = value

    return {
        "company_id": card["data-company-id"],                    # HTML 属性
        "name": card.select_one("h2 a").get_text(strip=True),     # 文本
        "industry": details.get("所属行业"),                       # 文本（dt/dd）
        "zone": details.get("所在区域"),                           # 文本（dt/dd）
        "device_count": int(details["设备数量"].replace("台", "").strip()),
        "status": card.select_one(".badge").get_text(strip=True),  # 文本
    }


companies = [parse_company_card(c) for c in cards]

print("本页企业数：", len(companies))
print(companies[0])


本页企业数： 20
{'company_id': 'C000001', 'name': '临港精工001有限公司', 'industry': '汽车零部件', 'zone': '智造一区', 'device_count': 6, 'status': '运营中'}


In [23]:
next_link = soup.select_one("nav.pagination a:last-child")

print(next_link.get_text(" ", strip=True))
print(next_link.get("href"))

下一页
/companies?page=2


In [24]:
from urllib.parse import urljoin

next_url = urljoin(current_url, next_link.get("href"))
print(next_url)

https://115.159.159.82/companies?page=2


In [27]:
companies = []
page_log = []                     # 每页条数，写报告用
base_url = "https://115.159.159.82"
current_url = base_url + "/companies"
page_no = 1

while current_url is not None and page_no <= 3:
    # 1) 请求当前页
    r = requests.get(current_url)
    soup = BeautifulSoup(r.text, "html.parser")

    # 2) 解析本页企业
    cards = soup.select("article.company-card")
    page_rows = [parse_company_card(c) for c in cards]
    companies.extend(page_rows)
    page_log.append(len(page_rows))
    print(f"第 {page_no} 页：{len(page_rows)} 条  ({current_url})")

    # 3) 找到下一页 href
    next_link = soup.select_one("nav.pagination a:last-child")
    if next_link is not None and next_link.get_text(strip=True) == "下一页":
        current_url = urljoin(current_url, next_link.get("href"))
        page_no += 1
    else:
        current_url = None

print("Company 总数：", len(companies))

第 1 页：20 条  (https://115.159.159.82/companies)
第 2 页：20 条  (https://115.159.159.82/companies?page=2)
第 3 页：20 条  (https://115.159.159.82/companies?page=3)
Company 总数： 60


In [28]:
print("companies:", len(companies))
print(
    "unique company_id:",
    len(set(x["company_id"] for x in companies))
)

print("devices:", len(devices))
print(
    "unique device_id:",
    len(set(x["device_id"] for x in devices))
)

companies: 60
unique company_id: 60
devices: 5000
unique device_id: 5000


In [34]:
from collections import Counter

device_counts = Counter(
    x["company_id"] for x in devices
)
mismatch_count = 0
mismatch_samples = []

for company in companies:
    cid = company["company_id"]
    expected = company["device_count"]          # 页面上写的设备数
    actual = device_counts.get(cid, 0)          # 设备表里实际的设备数

    if expected != actual:
        mismatch_count += 1
        if len(mismatch_samples) < 5:
            mismatch_samples.append((cid, expected, actual))
print("Company–Device 不一致企业数：", mismatch_count)

Company–Device 不一致企业数： 0


In [36]:
print("=" * 44)
print("Lab 1 最终验证结果")
print("=" * 44)
print("Company 总数              :", len(companies))
print("唯一 company_id 数        :", len(set(x["company_id"] for x in companies)))
print("Device 总数               :", len(devices))
print("唯一 device_id 数         :", len(set(x["device_id"] for x in devices)))
print("Company–Device 不一致企业数:", mismatch_count)
print("=" * 44)

Lab 1 最终验证结果
Company 总数              : 60
唯一 company_id 数        : 60
Device 总数               : 5000
唯一 device_id 数         : 5000
Company–Device 不一致企业数: 0


In [37]:
from pathlib import Path
import json

data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

with open(data_dir / "devices.json", "w", encoding="utf-8") as f:
    json.dump(devices, f, ensure_ascii=False, indent=2)

with open(data_dir / "companies.json", "w", encoding="utf-8") as f:
    json.dump(companies, f, ensure_ascii=False, indent=2)

print("数据已保存到 data/ 目录")

数据已保存到 data/ 目录
